<a href="https://colab.research.google.com/github/pesco13/QI452-Python-Quimica/blob/main/notebooks/04_PubChem_desde_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 04 · Consultar PubChem desde Python

**Objetivos**
* Entender qué es una API web.
* Obtener identificadores y propiedades de compuestos de PubChem con PUG-REST.

PubChem permite pedir datos escribiendo una dirección web con cuatro partes:

`https://pubchem.ncbi.nlm.nih.gov/rest/pug` + **entrada** (`/compound/name/cafeina`) + **operación** (`/property/MolecularWeight`) + **formato** (`/JSON`).

In [ ]:
import requests, time
import pandas as pd
BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

## 1. Del nombre al CID

El **CID** es el número de identificación de un compuesto en PubChem. Los nombres en inglés funcionan mejor.

In [ ]:
nombre = "caffeine"
r = requests.get(f"{BASE}/compound/name/{nombre}/cids/TXT")
print(r.status_code)   # 200 = todo bien
print(r.text)

## 2. Propiedades

Se pueden pedir varias separadas por comas. Nota: desde 2025 PubChem devuelve el SMILES con la clave `SMILES` (antes `CanonicalSMILES` / `IsomericSMILES`); la función de abajo acepta ambas.

In [ ]:
PROPS = "MolecularFormula,MolecularWeight,SMILES,XLogP,TPSA"

def propiedades_pubchem(nombre):
    url = f"{BASE}/compound/name/{nombre}/property/{PROPS}/JSON"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return {"nombre": nombre, "error": f"HTTP {r.status_code}"}
    datos = r.json()["PropertyTable"]["Properties"][0]
    return {
        "nombre": nombre,
        "CID": datos.get("CID"),
        "formula": datos.get("MolecularFormula"),
        "masa_molar": float(datos.get("MolecularWeight", "nan")),
        "smiles": datos.get("SMILES") or datos.get("IsomericSMILES") or datos.get("CanonicalSMILES"),
        "XLogP": datos.get("XLogP"),
        "TPSA": datos.get("TPSA"),
    }

propiedades_pubchem("caffeine")

## 3. Varios compuestos

PubChem permite hasta 5 consultas por segundo; `time.sleep` evita saturarlo.

In [ ]:
lista = ["vanillin", "methylparaben", "sorbic acid", "bisphenol A", "limonene", "compuesto_inventado"]
resultados = []
for n in lista:
    resultados.append(propiedades_pubchem(n))
    time.sleep(0.3)
pd.DataFrame(resultados)

La fila con error muestra por qué hay que revisar los datos descargados: un nombre mal escrito no devuelve nada.

## 4. Imagen de la estructura

In [ ]:
from IPython.display import Image
Image(url=f"{BASE}/compound/name/vanillin/PNG")

## Ejercicios

**1.** Obtenga la fórmula y la masa molar de tres solventes industriales (por ejemplo *toluene*, *ethyl acetate*, *hexane*).

<details><summary><b>Ver solución</b></summary>

```python
pd.DataFrame([propiedades_pubchem(n) for n in ["toluene", "ethyl acetate", "hexane"]])
```
</details>

**2.** Compare el XLogP de PubChem con el LogP de RDKit (notebook 03) para la vainillina. ¿Por qué no son idénticos?

<details><summary><b>Ver solución</b></summary>

```python
# Son dos modelos distintos de la misma propiedad: XLogP3 (PubChem) y Crippen (RDKit).
# Ambos son estimaciones, no valores experimentales.
```
</details>